In [3]:
import numpy as np
import pandas as pd
import os
from scipy.stats import kruskal
import calendar

def seasonality_analysis(df, col="VALOR", period=None, alpha=0.05):
    """
    Realiza análise de sazonalidade considerando apenas anos completos.

    Parâmetros
    ----------
    df : DataFrame
        DataFrame contendo pelo menos as colunas 'ano', 'mes' e `col`.
    col : str, default="VALOR"
        Coluna com valores mensais.
    period : tuple (ano_ini, ano_fim) ou None
        Período da análise. Se None, usa o período completo do arquivo (ano.min()..ano.max()).
    alpha : float, default=0.05
        Nível de significância para o teste de sazonalidade.

    Retorno
    -------
    dict com:
        - start_year : int
        - end_year : int
        - n_valid_years : int
        - invalid_years : str (anos com < 12 meses válidos, separados por vírgula)
        - significant : bool (se há diferenças sazonais entre meses)
        - p_value : float (teste de Kruskal-Wallis entre meses)
        - amplitude : float (diferença máx - mín entre médias mensais)
        - relative_strength : float (variância sazonal / variância total)
        - msi : float (Markham Seasonality Index, 0–1)
        - min_month : str (nome do mês com menor valor)
        - max_month : str (nome do mês com maior valor)
        - seasonal_index : dict {mes: índice relativo}
    """
    if not {"ano", "mes", col}.issubset(df.columns):
        raise ValueError("O DataFrame deve conter as colunas 'ano', 'mes' e a coluna de valores.")

    # define período
    if period is None:
        start_year = int(df["ano"].min())
        end_year   = int(df["ano"].max())
    else:
        start_year, end_year = map(int, period)

    # recorta período
    df_period = df.loc[(df["ano"] >= start_year) & (df["ano"] <= end_year)].copy()

    # identificar anos completos (12 meses válidos)
    valid_years = []
    invalid_years = []
    for year, group in df_period.groupby("ano"):
        if group[col].notna().sum() == 12:
            valid_years.append(year)
        else:
            invalid_years.append(year)

    n_valid_years = len(valid_years)
    invalid_years_str = ", ".join(map(str, invalid_years)) if invalid_years else ""

    # se não há anos completos, retorna vazio
    if n_valid_years == 0:
        return {
            "start_year": start_year,
            "end_year": end_year,
            "n_valid_years": 0,
            "invalid_years": invalid_years_str,
            "significant": False,
            "p_value": np.nan,
            "amplitude": np.nan,
            "relative_strength": np.nan,
            "msi": np.nan,
            "min_month": "",
            "max_month": "",
            "seasonal_index": {}
        }

    # filtra apenas anos válidos
    df_valid = df_period[df_period["ano"].isin(valid_years)].copy()

    # médias mensais
    monthly_means = df_valid.groupby("mes")[col].mean()

    # índice sazonal = desvio percentual em relação à média geral
    overall_mean = monthly_means.mean()
    seasonal_index = (monthly_means / overall_mean).to_dict()

    # amplitude (máx - mín)
    amplitude = float(monthly_means.max() - monthly_means.min())

    # força relativa da sazonalidade
    var_total = df_valid[col].var()
    var_seasonal = monthly_means.var()
    relative_strength = float(var_seasonal / var_total) if var_total > 0 else np.nan

    # meses críticos
    min_month_num = int(monthly_means.idxmin())
    max_month_num = int(monthly_means.idxmax())
    min_month = calendar.month_name[min_month_num]
    max_month = calendar.month_name[max_month_num]

    # teste de significância (Kruskal-Wallis entre meses)
    groups = [group[col].dropna().values for _, group in df_valid.groupby("mes")]
    if all(len(g) > 0 for g in groups):
        stat, p_value = kruskal(*groups)
        significant = bool(p_value < alpha)
    else:
        p_value = np.nan
        significant = False

    # MSI (Markham Seasonality Index)
    values = monthly_means.values
    if np.nansum(values) > 0:
        rel_freq = values / np.nansum(values)
        msi = np.sum(np.abs(rel_freq - 1/12)) / (2 * (1 - 1/12))
    else:
        msi = np.nan

    return {
        "start_year": start_year,
        "end_year": end_year,
        "n_valid_years": n_valid_years,
        "invalid_years": invalid_years_str,
        "significance": significant,
        "p_value": float(p_value),
        "amplitude": amplitude,
        "relative_strength": relative_strength,
        "msi": msi,
        "min_month": min_month,
        "max_month": max_month,
#        "seasonal_index": seasonal_index
    }

def seasonality_analysis_folder(folder, col="VALOR", period=None, file_ext=".csv"):
    """
    Aplica seasonality_analysis a todos os arquivos de uma pasta e consolida em um DataFrame.
    """
    results = []

    for file in os.listdir(folder):
        if file.endswith(file_ext):
            station_name = os.path.splitext(file)[0]
            filepath = os.path.join(folder, file)

            df = pd.read_csv(filepath)

            try:
                res = seasonality_analysis(df, col=col, period=period)
                res["station"] = station_name
                results.append(res)
            except Exception as e:
                print(f"Erro ao processar {file}: {e}")

    return pd.DataFrame(results)


#%%

folder_path = "/home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes/O3"
df_results = seasonality_analysis_folder(folder_path, col="VALOR")
print(df_results)

   start_year  end_year  n_valid_years  \
0        2005      2025             13   
1        2005      2025             11   
2        2005      2025             18   
3        2005      2025              9   
4        2005      2025             13   

                                       invalid_years  significance  \
0     2005, 2006, 2011, 2018, 2020, 2023, 2024, 2025          True   
1  2005, 2006, 2008, 2012, 2017, 2018, 2022, 2023...          True   
2                                   2005, 2007, 2025          True   
3  2005, 2006, 2007, 2011, 2015, 2016, 2019, 2020...          True   
4     2005, 2006, 2011, 2013, 2014, 2021, 2024, 2025          True   

        p_value  amplitude  relative_strength       msi min_month  max_month  \
0  2.259688e-10  15.808898           0.479123  0.081191   January  September   
1  8.469077e-07  13.393457           0.406338  0.076422  December     August   
2  2.227851e-05  30.181412           0.214915  0.059597   January     August   
3  2.2

In [4]:
# -*- coding: utf-8 -*-

from pathlib import Path
import json
import warnings
import calendar
import os

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.stats import kruskal

# -------------------------
# Configurações
# -------------------------
BASE_FOLDER = Path("/home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes")
STATIONS_FILE = Path("/home/nobre/Notebooks/RQAR_2025_book/data/Monitoramento_QAr_BR.csv")
OUTPUT_DIR = Path("/home/nobre/Notebooks/RQAR_2025_book/_static/preprocessed_seasonality")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POLLUTANTS = ["CO", "NO2", "SO2", "MP10", "MP25", "O3"]

ST_COL_ID = "ID_MMA_COMPLETO"
ST_COL_LAT = "LATITUDE"
ST_COL_LON = "LONGITUDE"
FILE_EXT = ".csv"
CRS_OUT = "EPSG:4326"

# -------------------------
# Função de análise de sazonalidade
# -------------------------

def seasonality_analysis(df, col="VALOR", period=None, alpha=0.05):
    """
    Realiza análise de sazonalidade considerando apenas anos completos.

    Entrada
    -----
    df : DataFrame com colunas 'ano', 'mes' e `col`.
    col : nome da coluna com valores (default 'VALOR').
    period : tuple (ano_ini, ano_fim) ou None -> período a considerar.
    alpha : nível de significância para Kruskal-Wallis.

    Retorna
    -----
    dicionário com métricas da análise de sazonalidade.
    """
    if not {"ano", "mes", col}.issubset(df.columns):
        raise ValueError("O DataFrame deve conter as colunas 'ano', 'mes' e a coluna de valores.")

    if period is None:
        start_year = int(df["ano"].min())
        end_year = int(df["ano"].max())
    else:
        start_year, end_year = map(int, period)

    df_period = df.loc[(df["ano"] >= start_year) & (df["ano"] <= end_year)].copy()

    # identificar anos completos (12 meses válidos)
    valid_years = []
    invalid_years = []
    for year, group in df_period.groupby("ano"):
        if group[col].notna().sum() == 12:
            valid_years.append(year)
        else:
            invalid_years.append(year)

    n_valid_years = len(valid_years)
    invalid_years_str = ", ".join(map(str, invalid_years)) if invalid_years else ""

    if n_valid_years == 0:
        return {
            "start_year": start_year,
            "end_year": end_year,
            "n_valid_years": 0,
            "invalid_years": invalid_years_str,
            "significance": False,
            "p_value": np.nan,
            "amplitude": np.nan,
            "relative_strength": np.nan,
            "msi": np.nan,
            "min_month": "",
            "max_month": "",
            "seasonal_index": {}
        }

    df_valid = df_period[df_period["ano"].isin(valid_years)].copy()

    monthly_means = df_valid.groupby("mes")[col].mean()
    overall_mean = monthly_means.mean()
    seasonal_index = (monthly_means / overall_mean).to_dict()

    amplitude = float(monthly_means.max() - monthly_means.min())

    var_total = df_valid[col].var()
    var_seasonal = monthly_means.var()
    relative_strength = float(var_seasonal / var_total) if var_total > 0 else np.nan

    min_month_num = int(monthly_means.idxmin())
    max_month_num = int(monthly_means.idxmax())
    min_month = calendar.month_name[min_month_num]
    max_month = calendar.month_name[max_month_num]

    groups = [group[col].dropna().values for _, group in df_valid.groupby("mes")]
    if all(len(g) > 0 for g in groups):
        try:
            stat, p_value = kruskal(*groups)
            significant = bool(p_value < alpha)
        except Exception:
            p_value = np.nan
            significant = False
    else:
        p_value = np.nan
        significant = False

    values = monthly_means.values
    if np.nansum(values) > 0:
        rel_freq = values / np.nansum(values)
        msi = np.sum(np.abs(rel_freq - 1/12)) / (2 * (1 - 1/12))
    else:
        msi = np.nan

    return {
        "start_year": start_year,
        "end_year": end_year,
        "n_valid_years": n_valid_years,
        "invalid_years": invalid_years_str,
        "significance": significant,
        "p_value": float(p_value) if not pd.isna(p_value) else np.nan,
        "amplitude": amplitude,
        "relative_strength": relative_strength,
        "msi": msi,
        "min_month": min_month,
        "max_month": max_month,
        "seasonal_index": seasonal_index
    }

# -------------------------
# Funções complementares
# -------------------------

def find_pollutant_folder(base_folder: Path, pol: str):
    if not base_folder.exists():
        return None
    for p in sorted(base_folder.iterdir()):
        if p.is_dir() and pol.lower() == p.name.lower():
            return p
        if p.is_dir() and pol.lower() in p.name.lower():
            return p
    return None


def read_stations_meta(stations_file: Path):
    try:
        df = pd.read_csv(stations_file)
    except Exception as e:
        warnings.warn(f"Não foi possível ler {stations_file}: {e}")
        return pd.DataFrame(columns=[ST_COL_ID, ST_COL_LAT, ST_COL_LON])
    for c in [ST_COL_ID, ST_COL_LAT, ST_COL_LON]:
        if c not in df.columns:
            df[c] = pd.NA
    df = df.drop_duplicates(subset=[ST_COL_ID])
    return df[[ST_COL_ID, ST_COL_LAT, ST_COL_LON]]


def merge_with_station_coords(season_df: pd.DataFrame, stations_meta: pd.DataFrame):
    if season_df.empty:
        return season_df
    if stations_meta.empty:
        season_df[ST_COL_LAT] = pd.NA
        season_df[ST_COL_LON] = pd.NA
        return season_df

    merged = season_df.merge(stations_meta, how="left", left_on="station", right_on=ST_COL_ID)
    if ST_COL_LAT not in merged.columns:
        merged[ST_COL_LAT] = pd.NA
    if ST_COL_LON not in merged.columns:
        merged[ST_COL_LON] = pd.NA
    return merged


def save_results_and_geojson(pol: str, merged_df: pd.DataFrame, out_dir: Path):
    pol_safe = pol.lower().replace(".", "").replace(" ", "_").replace(",", "")
    csv_path = out_dir / f"{pol_safe}_seasonality.csv"
    geojson_path = out_dir / f"{pol_safe}_stations.geojson"

    try:
        merged_df.to_csv(csv_path, index=False, encoding="utf-8")
    except Exception as e:
        print(f"Falha ao salvar CSV {csv_path}: {e}")

    n_points = 0
    if ST_COL_LAT in merged_df.columns and ST_COL_LON in merged_df.columns:
        mask = merged_df[ST_COL_LAT].notna() & merged_df[ST_COL_LON].notna()
        pts = merged_df.loc[mask].copy()
        if not pts.empty:
            try:
                pts[ST_COL_LAT] = pts[ST_COL_LAT].astype(float)
                pts[ST_COL_LON] = pts[ST_COL_LON].astype(float)
                pts["geometry"] = [Point(xy) for xy in zip(pts[ST_COL_LON], pts[ST_COL_LAT])]
                gdf = gpd.GeoDataFrame(pts, geometry="geometry", crs=CRS_OUT)
                gdf.to_file(geojson_path, driver="GeoJSON")
                n_points = len(gdf)
            except Exception as e:
                print(f"Falha ao gerar GeoJSON para {pol}: {e}")
                geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
                n_points = 0
        else:
            geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
            n_points = 0
    else:
        geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
        n_points = 0

    return {"pol": pol.lower(), "csv": csv_path.name, "geojson": geojson_path.name, "n_points": int(n_points)}

# -------------------------
# Runner principal
# -------------------------

def seasonality_analysis_folder(folder: Path, col="VALOR", period=None, file_ext=".csv"):
    results = []
    folder = Path(folder)
    if not folder.exists():
        return pd.DataFrame(results)

    for file in sorted(folder.iterdir()):
        if file.is_file() and file.suffix.lower() == file_ext:
            station_name = file.stem
            try:
                df = pd.read_csv(file)
            except Exception as e:
                print(f"Falha ao ler {file.name}: {e}")
                continue

            try:
                res = seasonality_analysis(df, col=col, period=period)
                res["station"] = station_name
                results.append(res)
            except Exception as e:
                print(f"Erro ao processar {file.name}: {e}")

    return pd.DataFrame(results)


def process_all_seasonality(base_folder: Path = BASE_FOLDER, stations_file: Path = STATIONS_FILE,
                            pollutants=POLLUTANTS, out_dir: Path = OUTPUT_DIR, file_ext=FILE_EXT,
                            col="VALOR", period=None):
    manifest = {"generated": []}
    print("Base folder:", base_folder)

    stations_meta = read_stations_meta(stations_file)

    for pol in pollutants:
        pol_folder = find_pollutant_folder(base_folder, pol)
        if pol_folder is None:
            print(f"Pasta do poluente '{pol}' não encontrada em {base_folder}; pulando.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        csv_files = [f for f in sorted(pol_folder.iterdir()) if f.is_file() and f.suffix.lower() == file_ext]
        if not csv_files:
            print(f"Pasta encontrada para '{pol}' ({pol_folder}), mas sem arquivos {file_ext}; pulando.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        print(f"Processando sazonalidade do poluente '{pol}' em: {pol_folder} (arquivos: {len(csv_files)})")

        season_df = seasonality_analysis_folder(pol_folder, col=col, period=period, file_ext=file_ext)

        if season_df.empty:
            print(f"Nenhum resultado válido para {pol} após análise; pulando geração de ficheiros.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        merged = merge_with_station_coords(season_df, stations_meta)

        summary = save_results_and_geojson(pol, merged, out_dir)
        manifest["generated"].append(summary)
        print(f"{pol}: CSV -> {summary['csv']}, GeoJSON -> {summary['geojson']} (pontos: {summary['n_points']})")

    manifest_path = out_dir / "manifest_seasonality.json"
    try:
        with open(manifest_path, "w", encoding="utf-8") as fh:
            json.dump(manifest, fh, ensure_ascii=False, indent=2)
        print("Manifest salvo em:", manifest_path)
    except Exception as e:
        print("Falha ao salvar manifest:", e)

    return manifest


# -------------------------
# Execução
# -------------------------
if __name__ == "__main__":
    manifest = process_all_seasonality()
    print("Concluído. Manifest:", manifest)


Base folder: /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes
Pasta do poluente 'CO' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes; pulando.
Pasta do poluente 'NO2' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes; pulando.
Pasta do poluente 'SO2' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes; pulando.
Pasta do poluente 'MP10' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes; pulando.
Pasta do poluente 'MP25' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes; pulando.
Processando sazonalidade do poluente 'O3' em: /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/mes/O3 (arquivos: 5)
O3: CSV -> o3_seasonality.csv, GeoJSON -> o3_stations.geojson (pontos: 5)
Manifest salvo em: /home/nobre/Notebooks/RQAR_2025_book/_static/preprocessed_seasonality/manifest_seasonality.json
Concluído. Manifest: {'generated': [{'pol': 'co', 'csv